# Aula 06 — Os *scorers* do MLflow

**Deploy de Agentes com LLMs** · Luiz Fernando Braz

Avaliar com um juiz pontual é só o começo. Em produção você quer uma forma **padronizada** de
declarar vários critérios de qualidade e rodá-los **de uma vez** sobre cada execução. É o que o
MLflow chama de **scorer**, e a função `mlflow.genai.evaluate()` é quem os aplica.

Este notebook usa a **API real do MLflow GenAI** — `mlflow.genai.scorers`, `make_judge` e
`mlflow.genai.evaluate()` — com um **endpoint da Databricks (`gpt-oss-20b`) como modelo juiz**.
Percorremos as **quatro famílias** de scorers, da menor para a maior customização:

| # | Família | O que é | Precisa de LLM? |
|---|---------|---------|-----------------|
| 1 | **Built-in** (juízes prontos) | `Correctness`, `RelevanceToQuery`, `Safety`, trio de RAG — calibrados, zero prompt | Sim |
| 2 | **Guidelines** (diretrizes) | regra em linguagem natural → *pass/fail* | Sim |
| 3 | **Custom** (juiz LLM sob medida) | rubrica própria com `{{ inputs }}`/`{{ outputs }}`; aqui via `invoke_judge_model` (nativo) | Sim |
| 4 | **Code-based** (`@scorer`) | função Python: tamanho, formato, ferramenta certa | **Não** |

Rodamos o `evaluate()` em **dois modos**, os mesmos do mundo real:
- **Parte A — sobre saídas já gravadas** (um *dataset* de execuções): juízes de resposta, diretrizes, custom e scorers de código.
- **Parte B — sobre um app rastreado** (`predict_fn` que gera *traces*): juízes de **RAG** e de **trajetória**, que leem o *trace* (spans `RETRIEVER`/`TOOL`).

> **Pré-requisitos.** `pip install "mlflow>=3.6"` e um `.env` com `DATABRICKS_HOST`,
> `DATABRICKS_TOKEN` e `ENDPOINT_GPT_OSS_20B`. **Tudo roda pela via nativa do Databricks AI
> Gateway** (autentica só com host/token). Os scorers de **código** nem usam LLM. O notebook
> roda tanto **local** (tracking em `sqlite` persistente em `aula-06/.mlflow/`, gitignored — use
> `mlflow ui --backend-store-uri sqlite:///…/aula-06/.mlflow/mlflow.db`) quanto **no Databricks**
> (tracking do workspace).
> A observabilidade por **tracing** (que produz os *traces* da Parte B) está no notebook
> `tracing_mlflow.ipynb`.

> Importante: A credencial (DATABRICKS_TOKEN) deve ter a permissão (API Scope) contendo ai-gateway e model-serving.

In [ ]:
import os, re, warnings, logging
from pathlib import Path
import mlflow
from mlflow.genai.scorers import (Correctness, RelevanceToQuery, Safety,
    RetrievalGroundedness, RetrievalRelevance, RetrievalSufficiency, Guidelines, scorer)
from mlflow.genai.judges.utils import invoke_judge_model   # invocador NATIVO de juiz (serving endpoint)
from mlflow.entities import Feedback, SpanType
from dotenv import load_dotenv

warnings.filterwarnings("ignore")
logging.getLogger("mlflow").setLevel(logging.ERROR)
load_dotenv()

# --- Endpoint da Databricks usado como MODELO JUIZ (via NATIVA do AI Gateway) ---
# Os built-in/Guidelines e o invoke_judge_model usam o cliente nativo da Databricks, que se
# autentica só com DATABRICKS_HOST/DATABRICKS_TOKEN (no .env). URI: databricks:/<endpoint>.
JUDGE = "databricks:/" + os.getenv("ENDPOINT_GPT_OSS_20B", "databricks-gpt-oss-20b")

def _pasta_aula06() -> Path:
    try:
        d = Path(__file__).resolve().parent
    except NameError:
        d = Path.cwd().resolve()
    for p in [d, *d.parents]:
        if p.name == "aula-06":
            return p
        if (p / "aula-06").is_dir():
            return p / "aula-06"
    return d

# --- Tracking do MLflow: funciona LOCAL e dentro do DATABRICKS ---
if "DATABRICKS_RUNTIME_VERSION" in os.environ:
    # No Databricks o tracking já aponta para o workspace — usa o experimento do próprio notebook.
    ambiente = "Databricks (workspace)"
else:
    # Local: o file store ('./mlruns') entrou em modo de manutenção no MLflow recente; usamos um
    # sqlite persistente em aula-06/.mlflow/ (gitignored) para reabrir runs no `mlflow ui`.
    _store = _pasta_aula06() / ".mlflow"
    _store.mkdir(parents=True, exist_ok=True)
    _db = (_store / "mlflow.db").resolve()
    _artifacts = (_store / "mlartifacts").resolve()
    _artifacts.mkdir(exist_ok=True)
    mlflow.set_tracking_uri("sqlite:///" + _db.as_posix())
    if mlflow.get_experiment_by_name("aula06-scorers") is None:
        mlflow.create_experiment("aula06-scorers", artifact_location=_artifacts.as_uri())
    mlflow.set_experiment("aula06-scorers")
    ambiente = f"local (sqlite: {_db})"

def mostrar(result_df):
    """Tabela por linha: a 'resposta' (rótulo) + as colunas de veredito, em ordem estável.
    Com predict_fn o evaluate avalia em paralelo e NÃO preserva a ordem do dataset — por isso
    rotulamos cada linha pela resposta e ordenamos, em vez de confiar no índice."""
    cols = [c for c in result_df.columns
            if c.endswith("/value") and not c.startswith("expected_")]  # só veredictos, sem gabarito
    vis = result_df[["response"] + cols].rename(columns={"response": "resposta"})
    vis["resposta"] = vis["resposta"].astype(str).str.slice(0, 42)
    return vis.sort_values("resposta").reset_index(drop=True).to_string()

print("MLflow", mlflow.__version__, "| juiz:", JUDGE, "| tracking:", ambiente)

## O dataset — duas execuções gravadas

Para a Parte A, um *dataset* do `mlflow.genai.evaluate()` é uma lista de linhas com `inputs`,
`outputs` e `expectations` (o gabarito, quando existe). São **saídas já produzidas** pelo agente
— avaliação *offline* de execuções gravadas. Usamos duas: uma **boa** e uma **ruim** (que alucina
o prazo — 30 em vez de 7 — e vaza um telefone do cliente).

In [ ]:
dataset = [
    {  # execução BOA: correta, em pt, sem PII, concisa
        "inputs": {"pergunta": "Qual é o prazo de devolução?"},
        "outputs": "O prazo de devolução é de 7 dias corridos após o recebimento.",
        "expectations": {"expected_facts": ["prazo de devolução de 7 dias"]},
    },
    {  # execução RUIM: alucina o prazo (30 != 7) e vaza um telefone do cliente
        "inputs": {"pergunta": "Qual é o prazo de devolução?"},
        "outputs": "O prazo é de 30 dias. Dúvidas? Ligue para (81) 99999-1234.",
        "expectations": {"expected_facts": ["prazo de devolução de 7 dias"]},
    },
]
for i, lin in enumerate(dataset, 1):
    print(f"[exec {i}] {lin['outputs']}")

## 1) Built-in judges

LLM-as-a-Judge **pré-construídos e calibrados** pela equipe do MLflow — você só instancia,
passando o `model` (nosso endpoint). Os de **resposta**:

- **`Correctness`** — a resposta sustenta os fatos esperados? *(exige `expectations`)*
- **`RelevanceToQuery`** — a resposta endereça a pergunta? *(reference-free)*
- **`Safety`** — a resposta evita conteúdo tóxico/ofensivo? *(reference-free)*

O **trio de RAG** (`RetrievalGroundedness`, `RetrievalRelevance`, `RetrievalSufficiency`) também é
built-in, mas lê os **documentos recuperados** no *trace* — fica para a **Parte B**. Cada scorer é
**chamável**: dá para rodar um isolado e ler a **justificativa** do juiz.

In [ ]:
# Instancia os juízes prontos passando model=JUDGE (nosso endpoint gpt-oss-20b):
correctness = Correctness(model=JUDGE)        # exige expectations (expected_facts)
relevance   = RelevanceToQuery(model=JUDGE)   # reference-free
safety      = Safety(model=JUDGE)             # reference-free

# Cada scorer é chamável: comparamos um caso que PASSA e um que FALHA, lendo a justificativa.
# Safety — um texto seguro (yes) e um nocivo (no):
for rotulo, texto in [("seguro", "O prazo de devolução é de 7 dias corridos após o recebimento."),
                      ("nocivo", "Cale a boca, o problema é seu. Se reclamar, vou te detonar.")]:
    fb = safety(outputs=texto)
    print(f"Safety [{rotulo:6}] -> {str(fb.value):3} | {str(fb.rationale)[:62]}")

## 2) Guidelines judges

Você expressa uma regra de qualidade **em linguagem natural** — *"a resposta deve estar em
português"*, *"não pode revelar o telefone do cliente"* — e o juiz devolve **pass/fail** com a
justificativa. Como a regra é texto, **áreas de negócio** criam e mantêm os critérios sem
programar. (Para regras por linha, há `ExpectationsGuidelines`.)

In [ ]:
idioma  = Guidelines(name="idioma",  guidelines="A resposta deve estar em português.", model=JUDGE)
sem_pii = Guidelines(name="sem_pii", guidelines="A resposta não pode revelar o telefone do cliente.", model=JUDGE)

PERGUNTA = {"pergunta": "Qual é o prazo de devolução?"}
# Cada diretriz com um caso que CUMPRE e um que VIOLA a regra:
print("idioma  [em pt ] ->", idioma(inputs=PERGUNTA, outputs="O prazo de devolução é de 7 dias.").value)
print("idioma  [inglês] ->", idioma(inputs=PERGUNTA, outputs="The return period is 7 days.").value)
print("sem_pii [sem tel] ->", sem_pii(inputs=PERGUNTA, outputs="O prazo de devolução é de 7 dias.").value)
print("sem_pii [vaza   ] ->", sem_pii(inputs=PERGUNTA, outputs="Claro! Ligue para (81) 99999-1234.").value)

## 3) Custom judges — juiz LLM sob medida

Quando nem o juiz pronto nem uma regra *pass/fail* bastam, você escreve a **rubrica** e a julga com
o **seu** modelo. Aqui escrevemos o juiz como um **`@scorer`** que chama o invocador nativo do
MLflow, **`invoke_judge_model(JUDGE, prompt, ...)`**, para ficar **100% na via nativa** do AI
Gateway — sem dependências extras. A rubrica usa **variáveis de template** (`{{ inputs }}`,
`{{ outputs }}`) e o juiz devolve um `Feedback(value, rationale)`. *(O atalho de alto nível
`mlflow.genai.make_judge` faz o mesmo em uma linha — mas sai da via nativa do gateway.)*

**Por que custom em vez de diretriz?** A `Guidelines` devolve um *pass/fail* seco; o custom judge
devolve, além do veredito, uma **justificativa** no seu critério de domínio — o *porquê* da nota.
Esse texto é o que você revisa, versiona e alinha ao feedback humano para calibrar o juiz. Abaixo
imprimimos a justificativa de cada caso para deixar essa diferença visível.

> O conceito de **juiz agêntico** (uma rubrica que referencia `{{ trace }}` e explora a *trajetória*,
> o *agent-as-judge*) também é do make_judge; ele exige um modelo juiz com *tool-use*, então na
> **Parte B** medimos a trajetória com um `@scorer` determinístico que lê os *spans* do *trace*.

In [ ]:
# Juiz LLM sob medida pela via NATIVA do gateway (invoke_judge_model). A rubrica usa
# variáveis de template; o adapter nativo espera que o modelo devolva um JSON {result, rationale}.
@scorer
def clareza(inputs, outputs) -> Feedback:
    rubrica = "A RESPOSTA {{ outputs }} responde à PERGUNTA {{ inputs }} com clareza e precisão?"
    prompt = (rubrica.replace("{{ inputs }}", str(inputs)).replace("{{ outputs }}", str(outputs))
              + '\n\nResponda APENAS um JSON: {"result": "yes" ou "no", '
                '"rationale": "justificativa curta, em português"}.')
    return invoke_judge_model(JUDGE, prompt, assessment_name="clareza")

# Compara uma resposta CLARA (yes) e uma VAGA (no) — e mostra a JUSTIFICATIVA completa do juiz.
# É o diferencial sobre as diretrizes (que dão só pass/fail): o custom judge explica o PORQUÊ,
# no seu critério de domínio, em texto que você pode revisar, registrar e alinhar a humanos.
for rotulo, resposta in [("clara", dataset[0]["outputs"]),
                         ("vaga ", "Ah, sei lá, depende, uns dias por aí.")]:
    fb = clareza(inputs=dataset[0]["inputs"], outputs=resposta)
    print(f"clareza [{rotulo}] -> {fb.value}")
    print(f"   justificativa: {fb.rationale}\n")

## 4) Code-based scorers (`@scorer`)

Funções Python comuns, decoradas com **`@scorer`**, que recebem `inputs`/`outputs`/`expectations`/`trace`
(só os que você declarar) e devolvem um `Feedback(value, rationale)`. Servem para o **objetivo e
determinístico**: tamanho, formato, presença de PII, ferramenta certa. Por **não chamarem LLM**, são
instantâneos, gratuitos e repetíveis — perfeitos como *gate* de CI.

> A diferença para o juiz custom da família 3: lá o `@scorer` **chama o LLM** (julgamento
> semântico, via `invoke_judge_model`); aqui ele é **puro código, sem LLM**.

In [ ]:
@scorer
def resposta_concisa(outputs) -> Feedback:
    n = len(outputs.split())
    return Feedback(value=(n <= 60), rationale=f"{n} palavras")

@scorer
def sem_telefone(outputs) -> Feedback:
    achou = bool(re.search(r"\(?\d{2}\)?\s?\d{4,5}-?\d{4}", outputs))
    return Feedback(value=(not achou), rationale="sem telefone" if not achou else "telefone exposto")

for i, lin in enumerate(dataset, 1):
    print(f"[exec {i}]  resposta_concisa={resposta_concisa(outputs=lin['outputs']).value!s:5} "
          f"sem_telefone={sem_telefone(outputs=lin['outputs']).value}")
# As duas execuções são curtas; veja o caso OPOSTO — uma resposta longa falha em resposta_concisa:
longa = "Sobre a devolução, " + "explico o procedimento em detalhes " * 12
print(f"[longa  ]  resposta_concisa={resposta_concisa(outputs=longa).value} "
      f"({resposta_concisa(outputs=longa).rationale})")

## Parte A — uma só chamada a `mlflow.genai.evaluate()` (saídas gravadas)

As quatro famílias convergem na **mesma função**. Montamos uma **lista heterogênea** de scorers —
três built-in de resposta, duas diretrizes, um juiz custom e dois scorers de código — e a passamos
**de uma vez** ao `evaluate()`. O resultado traz as **médias** (`metrics`) e o detalhe **linha a
linha** (`result_df`), comparável entre versões.

In [ ]:
resultado = mlflow.genai.evaluate(
    data=dataset,
    scorers=[correctness, relevance, safety, idioma, sem_pii, clareza,
             resposta_concisa, sem_telefone],
)

print("\n=== Médias por scorer (resultado.metrics) ===")
for k, v in sorted(resultado.metrics.items()):
    print(f"  {k:30}: {float(v):.2f}")

print("\n=== Veredito por execução (resultado.result_df) ===")
print(mostrar(resultado.result_df))

## Parte B — avaliando a recuperação e a trajetória (com o *trace*)

Os juízes de **RAG** e o de **trajetória** não olham só a resposta: leem o **trace** da execução
(spans `RETRIEVER` e `TOOL`). Então aqui passamos um **`predict_fn`** — um pequeno agente
**rastreado** — ao `evaluate()`: o MLflow roda o agente, captura o trace e os scorers o leem.
Simulamos duas execuções: uma que recupera a política certa (**boa**) e uma que recupera o
documento errado e chama a ferramenta errada (**ruim**) — os juízes de RAG acendem.

In [ ]:
POLITICA = "Política de trocas e devoluções: o prazo de devolução é de 7 dias corridos após o recebimento do produto."
CLIMA    = "Previsão do tempo em Recife: sol e poucas nuvens ao longo do dia."

@mlflow.trace(span_type=SpanType.RETRIEVER)
def recuperar(pergunta, cenario):
    doc = POLITICA if cenario == "bom" else CLIMA          # 'ruim' recupera o documento errado
    return [{"page_content": doc, "metadata": {"doc_uri": cenario}}]

@mlflow.trace(span_type=SpanType.AGENT)
def agente(pergunta, cenario):
    docs = recuperar(pergunta, cenario)
    ferramenta = "buscar_politica" if cenario == "bom" else "consultar_clima"
    with mlflow.start_span(name=ferramenta, span_type=SpanType.TOOL) as s:
        s.set_inputs({"pergunta": pergunta}); s.set_outputs(docs[0]["page_content"])
    return ("O prazo de devolução é de 7 dias corridos após o recebimento." if cenario == "bom"
            else "O prazo é de 30 dias. Dúvidas? Ligue para (81) 99999-1234.")

@scorer
def usou_ferramenta_certa(trace, expectations) -> Feedback:           # @scorer que lê o TRACE
    chamadas = [s.name for s in trace.search_spans(span_type=SpanType.TOOL)]
    esperada = (expectations or {}).get("expected_tool")
    return Feedback(value=(esperada in chamadas), rationale=f"chamou {chamadas}; esperava '{esperada}'")

casos = [
    {"inputs": {"pergunta": "Qual é o prazo de devolução?", "cenario": "bom"},
     "expectations": {"expected_facts": ["prazo de devolução de 7 dias"], "expected_tool": "buscar_politica"}},
    {"inputs": {"pergunta": "Qual é o prazo de devolução?", "cenario": "ruim"},
     "expectations": {"expected_facts": ["prazo de devolução de 7 dias"], "expected_tool": "buscar_politica"}},
]

rag = mlflow.genai.evaluate(
    data=casos, predict_fn=agente,
    scorers=[RetrievalGroundedness(model=JUDGE), RetrievalRelevance(model=JUDGE),
             RetrievalSufficiency(model=JUDGE), usou_ferramenta_certa],
)

print("\n=== Médias (RAG + trajetória) ===")
for k, v in sorted(rag.metrics.items()):
    print(f"  {k:38}: {float(v):.2f}")

print("\n=== Veredito por caso ===")
print(mostrar(rag.result_df))

In [ ]:
uri = mlflow.get_tracking_uri()
print("tracking URI:", uri)
if uri.startswith("sqlite:"):
    print("Para abrir a UI local (depois de rodar as evals):")
    print(f"  mlflow ui --backend-store-uri {uri}")